# IOAI — 2025 Lab Lab3 Pedestrian Detection (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
print('데이터(lab3.zip)는 노트북 셀이 공개 GCS 에서 자동 다운로드합니다.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Lab 3

This week we look at a new dataset below.

In [ ]:
!curl https://storage.googleapis.com/aiolympiadmy/ioai-2025-tsp/lab3.zip -o lab3.zip

In [ ]:
!unzip -nq lab3.zip

The above commands should create a `train/` folder and `test/` folder in this directory. Each folder contains `images/` and `labels/` subdirectories respectively.

In this dataset, each image is associated with a text file. In the text file, each row represents a single bounding box around a person in the image. Each bounding box is represented by 5 numbers: x_center, y_center, width, height, objectness. Objectness is always 1. All other dimensions are represented as fractions of image dimensions. e.g. x_center's actual pixel location needs to be multiplied by the width of the image.

If you still recall the network you worked with in Lab 2:
```python
class FCN(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)
        return x

backbone = ...
head = ...
model = FCN(backbone, head)
```

Here's an idea. Given that the backbone defined above will output a spatial feature map, we can iterate over each cell location in the spatial feature map, and predict if there is an object to detect in that cell (objectness). Notice that this is very similar to FCN segmentation where we were practically doing pixel-wise classification. However, we take this a bit further. If objectness in each cell is high enough, we predict an object within a bounding box as specified by our neural network.

## Another new network

Create a new class that is almost the exact same as `FCN` above, but specify the `head` to be a `nn.Conv2d` layer with kernel size 1. This layer should output 5 channels, corresponding to the 5 numbers in each row of our labels. This network will be architecturally simpler than the FCN of Lab 2!

_1 pt granted upon completion of network definition_

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Dataset and dataloaders

Now go ahead and build your Dataset and Dataloaders in Pytorch.

Note that the dataset should calculate and return x_offset and y_offset instead of x_center and y_center. If you leave x_center and y_center as is, you will force your network to also learn how to predict larger values of x_center and y_center with increasing values of x and y on the spatial feature map output by the backbone! You can resolve this by storing the offset between the bounding box center and the center of the cell of the spatial feature map your bounding box is in. I'll help you a little by providing an example:

```python
ipdb>  grid_size
8

ipdb>  grid_y
4

ipdb>  grid_y_min
0.5

ipdb>  y_center
0.5132275132275133

ipdb>  y_offset
0.013227513227513255
```

_1 pt granted upon successfully running the code below_,

```python
train_dataset = ...
test_dataset = ...
train_dataloader = DataLoader(train_dataset, ...)
test_dataloader = DataLoader(test_dataset, ...)
one_X, one_y = next(iter(test_dataset))
batch_X, batch_y = next(iter(test_dataloader))
```

_and demonstrating the following results_:
- `one_X.shape` = (3, im_h, im_w) where im_h and im_w are height and width of the image
- `one_y.shape` = (5, gy, gx) where gy and gx are height and width of the spatial feature map
- `batch_X.shape` = (B, 3, im_h, im_w) where B is batch size
- `batch_y.shape` = (B, 5, gy, gx)

Unless you've gone out of your way to resize your images to a shape other than a square, `im_h` should be equal to `im_w`, and `gy` should be equal to `gx`.

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Visualization

Being able to see is important for diagnosing computer vision applications. Create a visualization function to plot an overlay of bounding boxes on their respective images. You can use this function template below.

```python
def plot_batch_predictions(imgs, outputs):
    # img should have shape (batch, 3, im_h, im_w)
    # outputs should have shape (batch, 5, gy, gx)
    ...
```

Will also show you some sample matplotlib code to save time:

```python
import numpy as np
import matplotlib.pyplot as plt

# This creates a matplotlib figure with 4 cols
# Note that axes is an array of individual `ax`
batch_size = 4
fig, axes = plt.subplots(ncols=batch_size, figsize=(20, 8))
axes = axes if isinstance(axes, np.ndarray) else [axes]  # Handle batch_size=1

# This is how to plot an image
# Note that imshow requires image dimensions to be (H, W, 3)
# while Pytorch works with image dimensions (3, H, W)!
mock_tensor = torch.rand(3, 128, 128)
mock_np = mock_tensor.permute(1, 2, 0).contiguous().numpy()
# Usually you would just use plt.imshow where plt will grab the latest ax
# When you have as many as 4 in this example, specify which ax to use
ax = axes[0]
ax.imshow(img_np)
    
# This is how to draw rectangles using matplotlib
xmin = int((x_center - width / 2) * im_width)
xmax = int((x_center + width / 2) * im_width)
rect = plt.Rectangle(
    (xmin, ymin), width, height, 
    linewidth=2, edgecolor="red", facecolor="none"
)
# Add the rectangle to the Axes
ax.add_patch(rect)
```

Remember that your network output is x_offset and y_offset, need to convert them back to x_center and y_center!

_1 pt granted upon plotting one batch of images and labels from `test_dataloader` using `plot_batch_predictions`_

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Setting up loss calculations

Objectness is essentially a binary classification task, while predicting the correct bounding box is a regression task. 

Set up loss function calculations for objectness using binary cross entropy, and for bounding box localization using MSE. Create both losses in the same function for convenience, but return them separately instead of as a sum so they are easy to log later on. 

Concept-wise this is pretty straightforward. However, implementation-wise, you will need to place your tensors with great care. I'll help you a bit by providing you with this template below.

```python
def custom_loss(preds, targets):
    # both preds and targets should have shape (B, 5, gy, gx)
    # where B is batch size, gy and gx are spatial feature map h and w 
    ...
    return objectness_loss, localization_loss
```

_1 pt granted upon completion of loss function calculation_

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Model evaluation and baseline score

Create a `test_one_epoch` function that takes the model and the test dataloader as arguments. Calculate and return box IOU score (`torchvision.ops.box_iou`) in a dictionary like so:

```pycon
>>> metrics = test_one_epoch(model, test_dataloader)
>>> print(metrics)
{"miou": 0.005}
```

_1 pt granted upon implementing `test_one_epoch` and seeing the mean IOU score of the untrained model_

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Model training

Train your model on the training set. Track objectness loss and localization loss during training for every 10 minibatches (a). I will leave it up to choose how to combine your losses. 

At the end of every epoch, show metrics on both train (b) and test data (c), and plot prediction outputs of the first batch of the test dataset (d). Save the best performing model with the highest mean IOU score on test (e).

You don't need to run training for too long. I suspect <50 epochs will be sufficient.

_1 pt granted upon completion of (a) to (e)._

_Another 1 pt granted for exceeding 0.4 mean IOU on the test dataset._

In [ ]:
# Your work here
import os
import random
import torch.optim as optim

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## Post training

Create a plot that contains four subplots: image with true bounding boxes, image with predicted bounding boxes, predicted objectness over spatial feature map, true objectness over spatial feature map. Repeat this plot for a few images.

_1 pt granted upon completion of the above_

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


What learning task did you just perform in this notebook?

_1 pt granted upon finding the right answer_

\# Your answer here

A YOLO-style object detection.

This dataset is 18 years old this year. What is it called?

_1 pt granted upon finding the right answer_

\# Your answer here

[Penn-Fudan Database for Pedestrian Detection and Segmentation.](https://www.cis.upenn.edu/~jshi/ped_html/)

## EX: Going off track

Name a limitation of this training setup and briefly explain your reasoning.

_1 pt granted upon a satisfactory answer_

\# Your work here

One limitation is that each grid cell can predict at most one box. If the centers of two people fall into the same cell, the model can only ever detect one of them, so it will systematically miss objects in crowded scenes.

---

Show me how you can modify this training setup to attain better performance.

_2 pts granted upon successfully scoring at least +0.2 mean IOU higher than the score of the best model above. Partial credit to be granted at discretion. Bonus additional +1 pt to be granted for outstanding improvements_

In [ ]:
# TODO: 여기에 풀이를 작성하세요


$$
\text{CIoU} = \text{IoU} - \frac{\rho^2(\mathbf b - \mathbf b^{gt})}{c^2} - \alpha v
$$

where:

- $\mathbf b$: predicted bounding box.
- $\mathbf b^{gt}$: ground-truth bounding box.
- $\rho(\cdot)$: Euclidean distance between the centers.
- $c$: diagonal length of the smallest enclosing box.
- $v = \frac{4}{\pi^2} (\arctan \frac{w^{gt}}{h^{gt}} - \arctan \frac{w}{h})^2$.
- $\alpha = \frac{v}{(1 - \text{IoU})+v}$.
- $w$, $h$: width and height of the predicted bounding box.
- $w^{gt}$, $h^{gt}$: width and height of the ground-truth bounding box.

In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


In [ ]:
# TODO: 여기에 풀이를 작성하세요


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv', 'submission.zip', 'submission.jsonl', 'submission.json']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)